## The Dummy Problem: Teaching a Robot when to apply braking torque
Target: When sensor reads 2m, motor speed should be 10 rad/s

Note: Code cahnges in this notebook from the lecture are:
 - Update the animation code to initialize `epoch_text` before the `animate()` function to prevent the `NameError`.

### Imports

In [ ]:
try:
    import google.colab
    print("Running on Google Colab")
    !pip install -q numpy matplotlib pillow
except ModuleNotFoundError:
    print("Running locally")

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

%matplotlib inline

np.set_printoptions(
    linewidth=120,
    formatter={'float': lambda x: f"{0:8.4g}" if abs(x) < 1e-10 else f"{x:8.4g}"}
)

np.random.seed(0)

print("Setup complete!")

### 1. Initialize the Data and Parameters

In [ ]:
x = 2.0          # Sensor reading (input)
y_true = 10.0    # Target motor speed (output)

w = 3.0          # Initial random weight
b = 1.0          # Initial random bias
learning_rate = 0.1

epochs = 5       # Number of times we will loop the learning process

print(f"--- INITIAL STATE ---")
print(f"Input (x): {x}, Target (y): {y_true}")
print(f"Starting Weight: {w:.2f}, Starting Bias: {b:.2f}\n")

### 2. Printing the equations for review

In [ ]:
print("-- EQUATIONS ---")
print("Prediction (y_pred) = w * x + b")
print("Loss (MSE)          = 0.5 * (y_true - y_pred)^2")
print("Gradient w.r.t w (dw) = - (y_true - y_pred) * x")
print("Gradient w.r.t b (db) = - (y_true - y_pred) * 1\n")

### 3. The Training Loop

In [ ]:
# Store history for plotting the loss curve and animation
loss_history = []
w_history = []
b_history = []

for epoch in range(1, epochs + 1):
    print(f"--- EPOCH {epoch} ---")
    # Print current weights and biases for the epoch
    print(f"Current Weight (w): {w:.2f}, Current Bias (b): {b:.2f}")

    # Step 1: Forward Pass (Prediction)
    y_pred = (w * x) + b

    # Step 2: Loss Function (Mean Squared Error)
    # Multiplying by 0.5 to make the derivative cleaner
    loss = 0.5 * (y_true - y_pred)**2
    loss_history.append(loss) # Store loss
    w_history.append(w) # Store weights for animation
    b_history.append(b) # Store biases for animation

    # Step 3: Backpropagation (Calculate Gradients)
    error = y_true - y_pred
    dw = -error * x   # Gradient of Loss w.r.t weight
    db = -error * 1   # Gradient of Loss w.r.t bias

    # Step 4: Gradient Descent (Update Weights)
    w_new = w - (learning_rate * dw)
    b_new = b - (learning_rate * db)

    # Print the math for the students to see
    print(f"Prediction : ({w:.2f} * {x}) + {b:.2f} = {y_pred:.2f}")
    print(f"Loss       : 0.5 * ({y_true} - {y_pred:.2f})^2 = {loss:.4f}")
    print(f"Gradients  : dw = {dw:.2f}, db = {db:.2f}")
    print(f"Updates    : w_new = {w:.2f} - ({learning_rate} * {dw:.2f}) = {w_new:.2f}")
    print(f"             b_new = {b:.2f} - ({learning_rate} * {db:.2f}) = {b_new:.2f}\n")

    # Save the new weights for the next epoch
    w = w_new
    b = b_new

print(f"--- FINAL STATE AFTER {epochs} EPOCHS ---")
print(f"Final Weight: {w:.2f}, Final Bias: {b:.2f}")
print(f"Final Prediction: {(w * x) + b:.2f} (Target was {y_true})")

### 4. Prepare plots

In [ ]:
# Prepare for animation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left plot: Regression line and target
ax1.set_xlim(0, 4)
ax1.set_ylim(0, 15)
ax1.set_xlabel('Sensor Reading (x)')
ax1.set_ylabel('Motor Speed (y)')
ax1.set_title('Gradient Descent: Regression Line Evolution')
ax1.grid(True)

# Target point
ax1.plot(x, y_true, 'ro', markersize=8, label='Target Point')

# Initialize regression line with first epoch values
x_vals_line = np.array([0, 4])
y_vals_line = w_history[0] * x_vals_line + b_history[0]

line, = ax1.plot(
    x_vals_line,
    y_vals_line,
    'b-',
    lw=2,
    label='Predicted Line'
)

ax1.legend()

# Right plot: Loss curve
ax2.set_xlim(0, epochs + 1)
ax2.set_ylim(0, max(loss_history) * 1.1)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (Mean Squared Error)')
ax2.set_title('Gradient Descent: Loss Curve')
ax2.grid(True)

# Initialize with first loss point
loss_line, = ax2.plot(
    range(1, epochs + 1),
    loss_history,
    'g-o',
    label='Loss'
)

ax2.legend()

### 5. Animate the change in loss curve & convergence

In [ ]:
epoch_text = ax2.text(0.05, 0.95, "", transform=ax2.transAxes,
                      verticalalignment="top")

In [ ]:
def animate(i):
    current_w = w_history[i]
    current_b = b_history[i]

    # Update regression line
    x_vals_line = np.array([0, 4])
    y_vals_line = current_w * x_vals_line + current_b
    line.set_data(x_vals_line, y_vals_line)

    # Update loss curve
    loss_line.set_data(range(1, i + 2), loss_history[:i+1])
    epoch_text.set_text(f'Epoch: {i+1}/{epochs}\nLoss: {loss_history[i]:.4f}')

    return line, loss_line, epoch_text

ani = FuncAnimation(fig, animate, frames=epochs, blit=True, interval=500)

# To save the animation as a GIF
ani.save('gradient_descent_animation.gif', writer='pillow')

# To display the animation in Colab (optional, can be removed if only GIF is needed)
HTML(ani.to_jshtml())